In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from scipy.sparse import load_npz
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load data
train_tfidf = load_npz("train_tfidf.npz")
valid_tfidf = load_npz("valid_tfidf.npz")
test_tfidf = load_npz("test_tfidf.npz")

train_df = pd.read_csv("train_features.csv")
valid_df = pd.read_csv("valid_features.csv")
test_df = pd.read_csv("test_features.csv")

y_train = train_df["label"].astype(int).values
y_valid = valid_df["label"].astype(int).values
y_test = test_df["label"].astype(int).values

# Get numeric features
numeric_cols = train_df.columns.difference(["text", "tokens", "pos_seq", "label"])
X_train_extra = train_df[numeric_cols].astype(np.float64).values
X_valid_extra = valid_df[numeric_cols].astype(np.float64).values
X_test_extra = test_df[numeric_cols].astype(np.float64).values

# Use TruncatedSVD to reduce dimensionality
print("Applying dimensionality reduction...")
n_components = 300
svd = TruncatedSVD(n_components=n_components, random_state=42)
X_train_tfidf = svd.fit_transform(train_tfidf)
X_valid_tfidf = svd.transform(valid_tfidf)
X_test_tfidf = svd.transform(test_tfidf)

print(f"Reduced TF-IDF shape: {X_train_tfidf.shape}")
print(f"Extra features shape: {X_train_extra.shape}")
print(f"Explained variance ratio: {svd.explained_variance_ratio_.sum():.4f}")

# Scale features
scaler_tfidf = StandardScaler()
X_train_tfidf = scaler_tfidf.fit_transform(X_train_tfidf)
X_valid_tfidf = scaler_tfidf.transform(X_valid_tfidf)
X_test_tfidf = scaler_tfidf.transform(X_test_tfidf)

scaler_extra = StandardScaler()
X_train_extra = scaler_extra.fit_transform(X_train_extra)
X_valid_extra = scaler_extra.transform(X_valid_extra)
X_test_extra = scaler_extra.transform(X_test_extra)

# Dataset class
class TextDataset(Dataset):
    def __init__(self, tfidf_features, extra_features, labels):
        self.tfidf = torch.FloatTensor(tfidf_features)
        self.extra = torch.FloatTensor(extra_features)
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.tfidf[idx], self.extra[idx], self.labels[idx]

# CNN Model for text classification
class TextCNN(nn.Module):
    def __init__(self, input_size, extra_features_size, num_filters=100, filter_sizes=[3, 4, 5], dropout=0.5):
        super(TextCNN, self).__init__()
        
        self.input_size = input_size
        
        # Convolutional layers with different kernel sizes
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=1, out_channels=num_filters, kernel_size=fs)
            for fs in filter_sizes
        ])
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
        
        # Simplified FC layers
        total_filters = num_filters * len(filter_sizes)
        self.fc1 = nn.Linear(total_filters + extra_features_size, 128)
        self.fc2 = nn.Linear(128, 2)
        
        self.relu = nn.ReLU()
    
    def forward(self, tfidf, extra):
        # Reshape TF-IDF: (batch, input_size) -> (batch, 1, input_size)
        x = tfidf.unsqueeze(1)
        
        # Apply convolutions + ReLU + max pooling
        conv_outputs = []
        for conv in self.convs:
            conv_out = self.relu(conv(x))
            # Global max pooling
            pooled = torch.max(conv_out, dim=2)[0]
            conv_outputs.append(pooled)
        
        # Concatenate all conv outputs
        conv_concat = torch.cat(conv_outputs, dim=1)
        conv_concat = self.dropout(conv_concat)
        
        # Concatenate with extra features
        combined = torch.cat([conv_concat, extra], dim=1)
        
        # Fully connected layers
        x = self.relu(self.fc1(combined))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

# Training function
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for tfidf, extra, labels in loader:
        tfidf, extra, labels = tfidf.to(device), extra.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(tfidf, extra)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    return total_loss / len(loader), correct / total

# Evaluation function
def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for tfidf, extra, labels in loader:
            tfidf, extra, labels = tfidf.to(device), extra.to(device), labels.to(device)
            outputs = model(tfidf, extra)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return np.array(all_preds), np.array(all_labels)

# Train a single model configuration
def train_model(config, train_dataset, valid_dataset, input_size, extra_size):
    batch_size = config['batch_size']
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
    
    model = TextCNN(
        input_size=input_size,
        extra_features_size=extra_size,
        num_filters=config['num_filters'],
        filter_sizes=config['filter_sizes'],
        dropout=config['dropout']
    ).to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
    
    num_epochs = 30
    best_val_acc = 0
    patience = 5
    patience_counter = 0
    best_state = None
    
    for epoch in range(num_epochs):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
        val_preds, val_labels = evaluate(model, valid_loader)
        val_acc = accuracy_score(val_labels, val_preds)
        
        scheduler.step(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    
    # Restore best model
    model.load_state_dict(best_state)
    return model, best_val_acc

# Hyperparameter configurations to test
input_size = X_train_tfidf.shape[1]
extra_size = X_train_extra.shape[1]

train_dataset = TextDataset(X_train_tfidf, X_train_extra, y_train)
valid_dataset = TextDataset(X_valid_tfidf, X_valid_extra, y_valid)
test_dataset = TextDataset(X_test_tfidf, X_test_extra, y_test)

# Hyperparameter grid
configs = [
    # Vary number of filters
    {'num_filters': 50, 'filter_sizes': [3, 4, 5], 'dropout': 0.5, 'lr': 0.001, 'weight_decay': 0.0001, 'batch_size': 128},
    {'num_filters': 100, 'filter_sizes': [3, 4, 5], 'dropout': 0.5, 'lr': 0.001, 'weight_decay': 0.0001, 'batch_size': 128},
    {'num_filters': 150, 'filter_sizes': [3, 4, 5], 'dropout': 0.5, 'lr': 0.001, 'weight_decay': 0.0001, 'batch_size': 128},
    
    # Vary filter sizes
    {'num_filters': 100, 'filter_sizes': [2, 3, 4], 'dropout': 0.5, 'lr': 0.001, 'weight_decay': 0.0001, 'batch_size': 128},
    {'num_filters': 100, 'filter_sizes': [3, 4, 5, 6], 'dropout': 0.5, 'lr': 0.001, 'weight_decay': 0.0001, 'batch_size': 128},
    
    # Vary dropout
    {'num_filters': 100, 'filter_sizes': [3, 4, 5], 'dropout': 0.3, 'lr': 0.001, 'weight_decay': 0.0001, 'batch_size': 128},
    {'num_filters': 100, 'filter_sizes': [3, 4, 5], 'dropout': 0.7, 'lr': 0.001, 'weight_decay': 0.0001, 'batch_size': 128},
    
    # Vary learning rate
    {'num_filters': 100, 'filter_sizes': [3, 4, 5], 'dropout': 0.5, 'lr': 0.0005, 'weight_decay': 0.0001, 'batch_size': 128},
    {'num_filters': 100, 'filter_sizes': [3, 4, 5], 'dropout': 0.5, 'lr': 0.002, 'weight_decay': 0.0001, 'batch_size': 128},
    
    # Vary weight decay (L2 regularization)
    {'num_filters': 100, 'filter_sizes': [3, 4, 5], 'dropout': 0.5, 'lr': 0.001, 'weight_decay': 0.00001, 'batch_size': 128},
    {'num_filters': 100, 'filter_sizes': [3, 4, 5], 'dropout': 0.5, 'lr': 0.001, 'weight_decay': 0.001, 'batch_size': 128},
]

print("\n" + "="*50)
print("HYPERPARAMETER TUNING")
print("="*50)

best_overall_acc = 0
best_config = None
best_model = None

for i, config in enumerate(configs):
    print(f"\n[{i+1}/{len(configs)}] Testing configuration:")
    print(f"  num_filters={config['num_filters']}, filter_sizes={config['filter_sizes']}")
    print(f"  dropout={config['dropout']}, lr={config['lr']}, weight_decay={config['weight_decay']}")
    
    model, val_acc = train_model(config, train_dataset, valid_dataset, input_size, extra_size)
    
    print(f"  → Validation Accuracy: {val_acc:.4f}")
    
    if val_acc > best_overall_acc:
        best_overall_acc = val_acc
        best_config = config
        best_model = model
        print(f"  → New best configuration!")d

print("\n" + "="*50)
print("BEST HYPERPARAMETERS")
print("="*50)
print(f"Number of filters: {best_config['num_filters']}")
print(f"Filter sizes: {best_config['filter_sizes']}")
print(f"Dropout: {best_config['dropout']}")
print(f"Learning rate: {best_config['lr']}")
print(f"Weight decay (L2): {best_config['weight_decay']}")
print(f"Batch size: {best_config['batch_size']}")
print(f"Best Validation Accuracy: {best_overall_acc:.4f}")

# Final evaluation on test set
print("\n" + "="*50)
print("FINAL EVALUATION ON TEST SET")
print("="*50)

test_loader = DataLoader(test_dataset, batch_size=best_config['batch_size'], shuffle=False)
test_preds, test_labels = evaluate(best_model, test_loader)

test_acc = accuracy_score(test_labels, test_preds)
precision, recall, f1, support = precision_recall_fscore_support(test_labels, test_preds, average='weighted')
conf_matrix = confusion_matrix(test_labels, test_preds)

# Get per-class metrics
precision_per_class, recall_per_class, f1_per_class, _ = precision_recall_fscore_support(test_labels, test_preds, average=None)

print(f"\nTest Accuracy: {test_acc:.4f}")
print(f"Weighted Precision: {precision:.4f}")
print(f"Weighted Recall: {recall:.4f}")
print(f"Weighted F1-Score: {f1:.4f}")

print("\nPer-class metrics:")
print(f"Class 0 (Non-sarcastic) - Precision: {precision_per_class[0]:.4f}, Recall: {recall_per_class[0]:.4f}, F1: {f1_per_class[0]:.4f}")
print(f"Class 1 (Sarcastic) - Precision: {precision_per_class[1]:.4f}, Recall: {recall_per_class[1]:.4f}, F1: {f1_per_class[1]:.4f}")

print("\nConfusion Matrix:")
print(conf_matrix)

print("\nClassification Report:")
print(classification_report(test_labels, test_preds))

# Save results for report
print("\n" + "="*50)
print("SUMMARY FOR REPORT")
print("="*50)
print(f"""
Convolutional Neural Network (CNN)

Model Architecture:
- Input dimension after SVD: {input_size}
- Additional features: {extra_size}
- Convolutional layers: {len(best_config['filter_sizes'])} parallel 1D convolutions
- Filter sizes (kernel sizes): {best_config['filter_sizes']}
- Number of filters per size: {best_config['num_filters']}
- Total convolutional filters: {best_config['num_filters'] * len(best_config['filter_sizes'])}
- Pooling: Global max pooling on each filter output
- Fully connected layers: 2 layers (128 → 2 neurons)
- Activation: ReLU
- Dropout rate: {best_config['dropout']}
- Total trainable parameters: {sum(p.numel() for p in best_model.parameters()):,}

Hyperparameters Tested:
- Number of filters: [50, 100, 150]
- Filter sizes: [[2,3,4], [3,4,5], [3,4,5,6]]
- Dropout rates: [0.3, 0.5, 0.7]
- Learning rates: [0.0005, 0.001, 0.002]
- Weight decay (L2 regularization): [0.00001, 0.0001, 0.001]

Hyperparameters Chosen:
- Number of filters: {best_config['num_filters']}
- Filter sizes: {best_config['filter_sizes']}
- Dropout: {best_config['dropout']}
- Learning rate: {best_config['lr']}
- Weight decay: {best_config['weight_decay']}
- Batch size: {best_config['batch_size']}

Model Results:
Test Accuracy: {test_acc:.4f}
Precision: {precision:.4f}
Recall: {recall:.4f}
F1-Score: {f1:.4f}

Confusion Matrix:
{conf_matrix}
""")

Using device: cpu
Applying dimensionality reduction...
Reduced TF-IDF shape: (21464, 300)
Extra features shape: (21464, 59)
Explained variance ratio: 0.2644

HYPERPARAMETER TUNING

[1/11] Testing configuration:
  num_filters=50, filter_sizes=[3, 4, 5]
  dropout=0.5, lr=0.001, weight_decay=0.0001
  → Validation Accuracy: 0.7304
  → New best configuration!

[2/11] Testing configuration:
  num_filters=100, filter_sizes=[3, 4, 5]
  dropout=0.5, lr=0.001, weight_decay=0.0001
  → Validation Accuracy: 0.7360
  → New best configuration!

[3/11] Testing configuration:
  num_filters=150, filter_sizes=[3, 4, 5]
  dropout=0.5, lr=0.001, weight_decay=0.0001
  → Validation Accuracy: 0.7332

[4/11] Testing configuration:
  num_filters=100, filter_sizes=[2, 3, 4]
  dropout=0.5, lr=0.001, weight_decay=0.0001
  → Validation Accuracy: 0.7346

[5/11] Testing configuration:
  num_filters=100, filter_sizes=[3, 4, 5, 6]
  dropout=0.5, lr=0.001, weight_decay=0.0001
  → Validation Accuracy: 0.7374
  → New best